# ISTG1003 – Airbnb-priser i Amsterdam

**Prosjekt: regresjon og klassifikasjon**  
Datasett: *Airbnb Prices in European Cities*, Amsterdam (Kaggle, The Devastator).

Vi bruker enkle metoder fra pensum og undersøker:

1. **Regresjon:** I hvilken grad kan Airbnb-priser forutsies ved hjelp av avstand til sentrum, antall soverom og gjestekapasitet? Vi inkluderer også om observasjonen kommer fra ukedager eller helg.
2. **Klassifikasjon:** Kan vi klassifisere utleiene som *privat rom* eller *hel bolig* ut fra pris, antall soverom, gjestekapasitet og avstand til sentrum?
3. **Tilleggsanalyse:** Hvordan varierer prisene mellom ukedager og helger?

**Merk:** `realSum` angir pris i euro for to personer og to netter. Analysen beskriver observasjonene i dette historiske datasettet, ikke dagens Airbnb-priser.

## 1. Biblioteker og filplassering

Notatboken skal ligge i `notebooks/`. CSV-filene skal ligge i `data/`:

```text
Statistikk-Prosjekt/
├── data/
│   ├── amsterdam_weekdays.csv
│   └── amsterdam_weekends.csv
├── notebooks/
│   └── ISTG1003_Airbnb_samlet_analyse.ipynb
├── src/
└── figures/
```

Vi bruker kun `pandas`, `matplotlib` og `scikit-learn`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, ConfusionMatrixDisplay

# Fungerer enten Jupyter starter i prosjektmappen eller i notebooks/
ROOT = Path.cwd()
if not (ROOT / "data" / "amsterdam_weekdays.csv").exists():
    ROOT = ROOT.parent

print("Prosjektmappe:", ROOT)

## 2. Les inn datasettene

Vi markerer om hver observasjon kommer fra ukedager (`weekend = 0`) eller helger (`weekend = 1`) og slår datasettene sammen.

In [ ]:
weekdays = pd.read_csv(ROOT / "data" / "amsterdam_weekdays.csv")
weekends = pd.read_csv(ROOT / "data" / "amsterdam_weekends.csv")

weekdays["period"] = "Weekday"
weekends["period"] = "Weekend"
weekdays["weekend"] = 0
weekends["weekend"] = 1

df = pd.concat([weekdays, weekends], ignore_index=True)

print("Antall rader og kolonner:", df.shape)
print("\nAntall observasjoner per periode:")
print(df["period"].value_counts())
print("\nManglende verdier:")
print(df.isna().sum())

## 3. Utforskende dataanalyse

Vi ser på pris, avstand, soverom og kapasitet før vi bygger modellene.

In [ ]:
columns = ["realSum", "dist", "bedrooms", "person_capacity"]
print(df[columns].describe().round(2))

print("\nKorrelasjon med pris:")
print(df[columns].corr()["realSum"].round(3))

print("\nAntall observasjoner med pris over 2000 euro:")
print((df["realSum"] > 2000).sum())

In [ ]:
# Pris og avstand til sentrum
plt.figure(figsize=(8, 5))
plt.scatter(df["dist"], df["realSum"], alpha=0.25)
plt.xlabel("Avstand til sentrum (km)")
plt.ylabel("Pris for to netter (EUR)")
plt.title("Pris mot avstand til sentrum")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fordeling av priser
plt.figure(figsize=(8, 5))
plt.hist(df["realSum"], bins=50, edgecolor="black")
plt.xlabel("Pris for to netter (EUR)")
plt.ylabel("Antall observasjoner")
plt.title("Fordeling av Airbnb-priser")
plt.tight_layout()
plt.show()

**Kort tolkning:** Vi undersøker om prisfordelingen er skjev, om enkelte priser er svært høye, og om det ser ut til å være sammenheng mellom pris og avstand. Korrelasjon betyr ikke årsakssammenheng.

## 4. Sammenlign ukedager og helger

Vi sammenligner både gjennomsnitt og median. Median er nyttig når det finnes svært høye priser.

In [ ]:
price_stats = df.groupby("period")["realSum"].agg(["mean", "median", "min", "max"])
print("Priser per periode (EUR):")
print(price_stats.round(2))

medians = df.groupby("period")["realSum"].median()
difference = medians["Weekend"] - medians["Weekday"]
percentage = difference / medians["Weekday"] * 100
print(f"\nMedianforskjell: {difference:.2f} EUR ({percentage:.1f} %)")

print("\nAntall per romtype og periode:")
print(pd.crosstab(df["room_type"], df["period"]))
print("\nFordeling i prosent:")
print(pd.crosstab(df["room_type"], df["period"], normalize="columns").mul(100).round(1))

room_prices = df.groupby(["room_type", "period"])["realSum"].median().unstack()
print("\nMedianpris etter romtype (EUR):")
print(room_prices.round(2))

In [ ]:
# Medianpris for ukedager og helger
medians.reindex(["Weekday", "Weekend"]).plot(kind="bar", figsize=(7, 4))
plt.ylabel("Medianpris for to netter (EUR)")
plt.xlabel("Periode")
plt.title("Ukedager og helger")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Medianpris per romtype, fordelt på periode
room_prices[["Weekday", "Weekend"]].plot(kind="bar", figsize=(9, 5))
plt.ylabel("Medianpris for to netter (EUR)")
plt.xlabel("Romtype")
plt.title("Pris etter romtype og periode")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Kort tolkning:** En forskjell i samlet medianpris kan delvis henge sammen med at andelen *hele boliger* og *private rom* er forskjellig i de to periodene. Dette viser ikke nødvendigvis prisendringen for den samme boligen fra ukedag til helg. *Delte rom* er en liten gruppe og bør tolkes varsomt.

## 5. Multippel lineær regresjon

Vi forsøker å forutsi `realSum` ved hjelp av **avstand, soverom, gjestekapasitet og helg/ukedag**. Vi trener på 80 % av dataene og tester på 20 %. 

Vi evaluerer med **R²** (forklaringsgrad) og **MAE** (gjennomsnittlig absolutt prediksjonsfeil).

In [ ]:
X = df[["dist", "bedrooms", "person_capacity", "weekend"]]
y = df["realSum"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

regression = LinearRegression()
regression.fit(X_train, y_train)
y_pred = regression.predict(X_test)

print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} EUR")
print(f"Konstantledd: {regression.intercept_:.2f}")
print("Koeffisienter:")
for name, value in zip(X.columns, regression.coef_):
    print(f"  {name}: {value:.2f}")

In [ ]:
# Faktisk pris sammenlignet med modellens prediksjoner
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Faktisk pris (EUR)")
plt.ylabel("Forutsagt pris (EUR)")
plt.title("Lineær regresjon – faktisk og forutsagt pris")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual = faktisk pris - forutsagt pris
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Forutsagt pris (EUR)")
plt.ylabel("Residual (EUR)")
plt.title("Residualanalyse")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Kort tolkning:** En positiv `weekend`-koeffisient betyr at modellen, med de andre variablene holdt konstante, forutsier en høyere pris for helgeobservasjoner. Residualene brukes til å diskutere feil og mulige svakheter ved den lineære modellen. Store ekstreme priser kan påvirke modellen.

## 6. Klassifikasjon med KNN

Vi bruker **KNN med fem naboer** til å skille mellom **privat rom** og **hel bolig**. De få observasjonene av *delt rom* holdes utenfor, slik at vi får to klasser.

Variablene standardiseres ut fra **treningsdataene**, fordi KNN bruker avstander mellom datapunkter.

In [ ]:
classification_df = df[df["room_type"].isin(["Private room", "Entire home/apt"])]

X = classification_df[["realSum", "bedrooms", "person_capacity", "dist"]]
y = classification_df["room_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("Antall observasjoner:", len(classification_df))
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.1f} %")

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["Private room", "Entire home/apt"],
    display_labels=["Privat rom", "Hel bolig"],
    cmap="Blues",
    colorbar=False
)
plt.title("KNN – klassifikasjon av romtype")
plt.tight_layout()
plt.show()

## 7. Oppsummering og diskusjon

**Resultater fra gruppens tidligere kjøring** (kontroller at dere får samme tall når dere kjører notatboken):

- Medianpris: **430,25 € på ukedager**, **491,65 € i helger** – en forskjell på **14,3 %**.
- Andel hele boliger: **48,8 % på ukedager**, **60,2 % i helger**. Dette kan være en del av forklaringen på den samlede prisforskjellen.
- Regresjon på de kombinerte dataene: **R² = 0,321**, **MAE = 194,79 €**. Koeffisienten for `weekend` var **+48,93 €**.
- KNN på de kombinerte dataene: **84,1 % accuracy**.

**Viktige begrensninger:**

- Datasettet inneholder noen svært høye priser, som kan påvirke regresjonen.
- Observasjonene fra ukedager og helger trenger ikke å være de samme boligene. Dermed kan vi ikke si at en bestemt bolig blir dyrere i helgen.
- Samme utleie kan eventuelt forekomme i begge datafilene, så trenings- og testobservasjoner kan være mindre uavhengige enn antatt. Datasettene inneholder ikke en tydelig unik bolig-ID som vi har brukt til å kontrollere dette.
- Resultatene viser **sammenhenger**, ikke årsak og virkning.

**Konklusjon:** Vi har undersøkt prisforskjeller mellom perioder, brukt lineær regresjon til å forutsi priser og KNN til å klassifisere romtyper. Modellene gir nyttig informasjon, men har klare begrensninger.

**Kilder og hjelpemidler:** Oppgi kilden til Kaggle-datasettet og deklarer KI-bruk i den endelige rapporten i samsvar med oppgaveteksten.